# Chronos-T5 Base (200M) — M6 Round 1 inference pilot

**Purpose.** Test whether `amazon/chronos-t5-base` can be run on the Stage 3
Round 1 context and whether its **raw** sampled forecasts can be saved. If the
run succeeds this may become an actual research result, so the implementation is
deliberately simple and reproducible.

**Experiment**

| Item | Value |
|---|---|
| Model | `amazon/chronos-t5-base` (Chronos-T5 Base, 200M parameters) |
| Round | M6 Round 1 |
| Forecast origin | 2022-03-04 (Friday) |
| Context | 512 weekday daily log returns ending on the origin |
| Assets | 100 official M6 assets, official order |
| Forecast horizon | 20 weekdays (2022-03-07 → 2022-04-01) |
| Trajectories | 100 sampled paths per asset |
| Raw output shape | `(100 assets, 100 samples, 20 steps)` |

**Chronos is univariate.** Each of the 100 assets is forecast as its own
independent series. Assets are passed to the model in batches purely for speed —
batching never lets one asset's history inform another's forecast, and it does
not change the asset ordering or the output shape.

**No preprocessing happens here.** The Round 1 context was produced in Stages 2
and 3 (shared weekday calendar, forward-filled closure days, DRE's official
zero-return treatment, log returns, 512-row slice). This notebook loads those
values and passes them to the model unchanged: no normalising, standardising,
smoothing, clipping, averaging or NaN-filling. CARR's and OGN's genuine leading
`NaN` values stay `NaN` — Chronos masks missing values natively through its
attention mask.

**Scope.** This notebook stops at the raw sampled trajectories. It does **not**
sum the 20 returns, convert to four-week returns, rank assets, build M6 quintile
probabilities, load realised outcomes, or compute RPS.

**Run the sections in order, top to bottom.**

In [2]:
!nvidia-smi

Wed Aug  5 12:52:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   43C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Install Chronos

Installs the stable `chronos-forecasting` package. The pretrained weights are
**not** downloaded here — they are fetched from the Hugging Face Hub in
section 6, into the Colab runtime's cache, never into the research repository.

Colab may ask you to restart the runtime after installing. If it does, restart
and then re-run from this cell.

In [3]:
%pip install -q chronos-forecasting

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.6/80.6 kB 8.3 MB/s eta 0:00:00


## 2. Imports and settings

The single settings block for the experiment. `SERIES_BATCH_SIZE` is the only
value you should need to change if the Colab GPU runs out of memory — it affects
speed and memory only, never the 100 assets, the 100 trajectories, the 20-step
horizon, or the asset ordering.

In [4]:
import hashlib
import json
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from chronos import ChronosPipeline

# --- Experiment settings ---------------------------------------------------
MODEL_ID = "amazon/chronos-t5-base"
ROUND_NUMBER = 1
ORIGIN_DATE = "2022-03-04"
CONTEXT_LENGTH = 512
PREDICTION_LENGTH = 20
NUM_SAMPLES = 100
RANDOM_SEED = 42

# Assets per forward pass. Lower this if Colab reports a CUDA OOM error.
# It changes speed/memory ONLY - never the results' shape or ordering.
SERIES_BATCH_SIZE = 10

# Round 1 forecast weekdays: 2022-03-07 .. 2022-04-01 (20 shared weekdays).
FORECAST_START_DATE = "2022-03-07"
FORECAST_END_DATE = "2022-04-01"

# Official M6 asset order - used to verify the loaded context, not to reorder it.
OFFICIAL_ASSET_ORDER = [
    "ABBV", "ACN", "AEP", "AIZ", "ALLE", "AMAT", "AMP", "AMZN", "AVB", "AVY",
    "AXP", "BDX", "BF-B", "BMY", "BR", "CARR", "CDW", "CE", "CHTR", "CNC",
    "CNP", "COP", "CTAS", "CZR", "DG", "DPZ", "DRE", "DXC", "EWA", "EWC",
    "EWG", "EWH", "EWJ", "EWL", "EWQ", "EWT", "EWU", "EWY", "EWZ", "FTV",
    "GOOG", "GPC", "GSG", "HIG", "HIGH.L", "HST", "HYG", "IAU", "ICLN",
    "IEAA.L", "IEF", "IEFM.L", "IEMG", "IEUS", "IEVL.L", "IGF", "INDA",
    "IUMO.L", "IUVL.L", "IVV", "IWM", "IXN", "JPEA.L", "JPM", "KR", "LQD",
    "MCHI", "META", "MVEU.L", "OGN", "PG", "PPL", "PRU", "PYPL", "RE",
    "REET", "ROL", "ROST", "SEGA.L", "SHY", "SLV", "SPMV.L", "TLT", "UNH",
    "URI", "V", "VRSK", "VXX", "WRK", "XLB", "XLC", "XLE", "XLF", "XLI",
    "XLK", "XLP", "XLU", "XLV", "XLY", "XOM",
]
N_ASSETS = len(OFFICIAL_ASSET_ORDER)

print(f"chronos-forecasting {version('chronos-forecasting')} | torch {torch.__version__}")
print(f"Experiment: {MODEL_ID} | round {ROUND_NUMBER} | origin {ORIGIN_DATE}")
print(f"Target raw output shape: ({N_ASSETS}, {NUM_SAMPLES}, {PREDICTION_LENGTH})")

chronos-forecasting 2.3.1 | torch 2.11.0+cu128
Experiment: amazon/chronos-t5-base | round 1 | origin 2022-03-04
Target raw output shape: (100, 100, 20)


## 3. Mount Google Drive

Colab cannot see the local VS Code repository, so Google Drive supplies the
Round 1 context file and stores the outputs permanently (Colab's own disk is
wiped when the runtime ends).

Running this cell opens a Google authorisation prompt.

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 4. Input and output paths

**Edit `DRIVE_PROJECT_DIR` below — it is the only path you need to change.**

Before running, copy the Round 1 context into that Drive folder so this layout
exists:

```
<DRIVE_PROJECT_DIR>/
└── data/processed/rolling_origins/round_01_context.csv   ← copy from the repo
```

Outputs are written to `<DRIVE_PROJECT_DIR>/outputs/chronos_t5_base/`, which is
created automatically.

In [8]:
# >>> EDIT THIS ONE LINE <<<
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/HonoursResearch/Round_1_Context")

CONTEXT_PATH = DRIVE_PROJECT_DIR / f"round_{ROUND_NUMBER:02d}_context.csv"

OUTPUT_DIR = DRIVE_PROJECT_DIR / "outputs" / "chronos_t5_base"
SAMPLES_PATH = OUTPUT_DIR / f"chronos_t5_base_round{ROUND_NUMBER:02d}_samples.npz"
METADATA_PATH = OUTPUT_DIR / f"chronos_t5_base_round{ROUND_NUMBER:02d}_metadata.json"
REPORT_PATH = OUTPUT_DIR / f"chronos_t5_base_round{ROUND_NUMBER:02d}_inference_report.md"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not CONTEXT_PATH.is_file():
    raise FileNotFoundError(
        f"Round {ROUND_NUMBER} context not found at:\n  {CONTEXT_PATH}\n"
        "Copy round_01_context.csv from the repository into that Drive folder, "
        "or correct DRIVE_PROJECT_DIR above."
    )

print(f"Context file : {CONTEXT_PATH}")
print(f"Output folder: {OUTPUT_DIR}")

Context file : /content/drive/MyDrive/HonoursResearch/Round_1_Context/round_01_context.csv
Output folder: /content/drive/MyDrive/HonoursResearch/Round_1_Context/outputs/chronos_t5_base


## 5. Load and validate the Round 1 context

Confirms the correct research input is being used — 512 rows, 100 assets in
official order, ending exactly on the forecast origin with nothing after it —
and then builds the model input in batch-first form, `(100 assets, 512 steps)`.

The final assertion proves the model input is a plain transpose of the loaded
values: nothing was rescaled, filled or otherwise altered. Leading `NaN`s for
CARR and OGN are expected and are passed through to the model as missing values.

In [9]:
context_df = pd.read_csv(CONTEXT_PATH, parse_dates=["date"])

# --- Structural validation --------------------------------------------------
assert context_df.shape[0] == CONTEXT_LENGTH, (
    f"Expected {CONTEXT_LENGTH} rows, found {context_df.shape[0]}"
)
assert list(context_df.columns) == ["date"] + OFFICIAL_ASSET_ORDER, (
    "Column names/order do not match 'date' + the official M6 asset order"
)
dates = context_df["date"]
assert dates.is_monotonic_increasing, "Context dates are not ascending"
assert not dates.duplicated().any(), "Context contains duplicate dates"
assert dates.iloc[-1] == pd.Timestamp(ORIGIN_DATE), (
    f"Context ends on {dates.iloc[-1].date()}, expected {ORIGIN_DATE}"
)
assert (dates <= pd.Timestamp(ORIGIN_DATE)).all(), "Context contains a date after the origin"

# --- Model input: (100 assets, 512 time steps), values untouched ------------
context_matrix = context_df[OFFICIAL_ASSET_ORDER].to_numpy(dtype=np.float64).T
assert context_matrix.shape == (N_ASSETS, CONTEXT_LENGTH)
assert np.array_equal(
    context_matrix, context_df[OFFICIAL_ASSET_ORDER].to_numpy().T, equal_nan=True
), "Model input differs from the loaded context values"

# --- Leading missing history (must remain NaN) ------------------------------
leading_missing = {}
for i, symbol in enumerate(OFFICIAL_ASSET_ORDER):
    row = context_matrix[i]
    n_lead = int(np.argmax(~np.isnan(row))) if np.isnan(row[0]) else 0
    if n_lead:
        leading_missing[symbol] = n_lead
assert set(leading_missing) == {"CARR", "OGN"}, (
    f"Unexpected assets with leading missing history: {sorted(leading_missing)}"
)

# --- Forecast dates ---------------------------------------------------------
FORECAST_DATES = pd.bdate_range(FORECAST_START_DATE, FORECAST_END_DATE)
assert len(FORECAST_DATES) == PREDICTION_LENGTH

CONTEXT_SHA256 = hashlib.sha256(CONTEXT_PATH.read_bytes()).hexdigest()

print("Round 1 context validated")
print(f"  model input shape : {context_matrix.shape}  (assets x time steps)")
print(f"  context start     : {dates.iloc[0].date()}")
print(f"  context end       : {dates.iloc[-1].date()}  (forecast origin)")
print(f"  assets            : {N_ASSETS} in official M6 order")
print(f"  leading NaN kept  : " + ", ".join(f"{k} ({v})" for k, v in leading_missing.items()))
print(f"  forecast dates    : {FORECAST_DATES[0].date()} .. {FORECAST_DATES[-1].date()} "
      f"({len(FORECAST_DATES)} weekdays)")

Round 1 context validated
  model input shape : (100, 512)  (assets x time steps)
  context start     : 2020-03-19
  context end       : 2022-03-04  (forecast origin)
  assets            : 100 in official M6 order
  leading NaN kept  : CARR (1), OGN (302)
  forecast dates    : 2022-03-07 .. 2022-04-01 (20 weekdays)


## 6. GPU check and model loading

> ⚠️ **Running this cell will download/load the pretrained Chronos model and
> place it on the Colab GPU.** The first run downloads roughly 200M parameters
> from the Hugging Face Hub into the runtime cache (not into your repository).

A GPU is required: if none is attached the cell stops with instructions rather
than silently running the 200M model on CPU. The model is used strictly for
inference — no training or fine-tuning.

In [10]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected. In Colab choose Runtime > Change runtime type > "
        "Hardware accelerator: GPU (T4 or better), then re-run this notebook "
        "from section 2. Refusing to run the 200M model on CPU."
    )

DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

pipeline = ChronosPipeline.from_pretrained(
    MODEL_ID,
    device_map="cuda",
    torch_dtype=DTYPE,
)

print(f"GPU   : {torch.cuda.get_device_name(0)}")
print(f"dtype : {DTYPE}")
print(f"Loaded: {MODEL_ID} (inference only, no fine-tuning)")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  806MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

GPU   : NVIDIA L4
dtype : torch.bfloat16
Loaded: amazon/chronos-t5-base (inference only, no fine-tuning)


## 7. Run inference

Each asset is forecast independently as its own univariate series. Assets are
grouped into batches of `SERIES_BATCH_SIZE` for speed; each batch element is a
separate 512-step series, so no asset's history can influence another's forecast.

Chronos handles the leading `NaN`s natively — its tokenizer marks missing values
in the attention mask, so CARR and OGN are forecast from their genuine history
without any padding being invented here.

Batch results are concatenated in the original asset order, giving the raw
`(100, 100, 20)` array. Expect a few minutes on a T4.

In [12]:
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

batch_outputs = []
for start in range(0, N_ASSETS, SERIES_BATCH_SIZE):
    stop = min(start + SERIES_BATCH_SIZE, N_ASSETS)
    # One independent 1-D series per asset in this batch.
    batch_context = [
        torch.tensor(context_matrix[i], dtype=torch.float32) for i in range(start, stop)
    ]
    samples = pipeline.predict(
    batch_context,
    prediction_length=PREDICTION_LENGTH,
    num_samples=NUM_SAMPLES,
)
    batch_outputs.append(samples.to(torch.float32).cpu().numpy())
    print(f"  assets {start + 1:3d}-{stop:3d} done -> {tuple(batch_outputs[-1].shape)}")

forecast_samples = np.concatenate(batch_outputs, axis=0)

# --- Validate the raw output ------------------------------------------------
assert forecast_samples.shape == (N_ASSETS, NUM_SAMPLES, PREDICTION_LENGTH), (
    f"Expected ({N_ASSETS}, {NUM_SAMPLES}, {PREDICTION_LENGTH}), "
    f"got {forecast_samples.shape}"
)
assert np.isfinite(forecast_samples).all(), "Forecasts contain NaN or infinite values"

print(f"\nRaw forecast array: {forecast_samples.shape} "
      "(assets x sampled trajectories x forecast weekdays)")
print("All values finite. No post-processing applied.")

  assets   1- 10 done -> (10, 100, 20)
  assets  11- 20 done -> (10, 100, 20)
  assets  21- 30 done -> (10, 100, 20)
  assets  31- 40 done -> (10, 100, 20)
  assets  41- 50 done -> (10, 100, 20)
  assets  51- 60 done -> (10, 100, 20)
  assets  61- 70 done -> (10, 100, 20)
  assets  71- 80 done -> (10, 100, 20)
  assets  81- 90 done -> (10, 100, 20)
  assets  91-100 done -> (10, 100, 20)

Raw forecast array: (100, 100, 20) (assets x sampled trajectories x forecast weekdays)
All values finite. No post-processing applied.


## 8. Save and verify the raw forecasts

The untransformed `(100, 100, 20)` array is written to Drive immediately, before
any analysis, so nothing is lost when the runtime ends. The saved file is then
reloaded and checked for shape and asset ordering.

In [13]:
asset_symbols = np.array(OFFICIAL_ASSET_ORDER)
forecast_date_strings = np.array([d.strftime("%Y-%m-%d") for d in FORECAST_DATES])

np.savez_compressed(
    SAMPLES_PATH,
    forecast_samples=forecast_samples,
    asset_symbols=asset_symbols,
    forecast_dates=forecast_date_strings,
)

metadata = {
    "model_id": MODEL_ID,
    "round": ROUND_NUMBER,
    "context_start_date": str(context_df["date"].iloc[0].date()),
    "context_end_date": str(context_df["date"].iloc[-1].date()),
    "num_assets": int(N_ASSETS),
    "num_samples": int(NUM_SAMPLES),
    "prediction_length": int(PREDICTION_LENGTH),
    "random_seed": int(RANDOM_SEED),
    "output_shape": list(forecast_samples.shape),
    "chronos_forecasting_version": version("chronos-forecasting"),
    "torch_version": torch.__version__,
    "dtype": str(DTYPE),
    "context_file_sha256": CONTEXT_SHA256,
    "run_timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

# --- Reload and verify ------------------------------------------------------
with np.load(SAMPLES_PATH, allow_pickle=False) as reloaded:
    reloaded_samples = reloaded["forecast_samples"]
    reloaded_symbols = reloaded["asset_symbols"]
    reloaded_dates = reloaded["forecast_dates"]

assert reloaded_samples.shape == (N_ASSETS, NUM_SAMPLES, PREDICTION_LENGTH)
assert list(reloaded_symbols) == OFFICIAL_ASSET_ORDER, "Asset ordering changed on save/reload"
assert np.array_equal(reloaded_samples, forecast_samples), "Saved values differ from the forecasts"
assert len(reloaded_dates) == PREDICTION_LENGTH

SAVE_VERIFIED = True
print(f"Saved   : {SAMPLES_PATH}")
print(f"Metadata: {METADATA_PATH}")
print(f"Reloaded: {reloaded_samples.shape}, asset order unchanged, values identical.")

Saved   : /content/drive/MyDrive/HonoursResearch/Round_1_Context/outputs/chronos_t5_base/chronos_t5_base_round01_samples.npz
Metadata: /content/drive/MyDrive/HonoursResearch/Round_1_Context/outputs/chronos_t5_base/chronos_t5_base_round01_metadata.json
Reloaded: (100, 100, 20), asset order unchanged, values identical.


## 9. Write the inference report

Writes a short report describing what was actually run. It is saved next to the
outputs in Drive; copy it into the repository as
`reports/chronos_t5_base_round01_inference_report.md` after the run.

In [14]:
report = f"""# Chronos-T5 Base (200M) - M6 Round 1 Inference Report

Generated: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}

## Experiment

- Model: `{MODEL_ID}` (Chronos-T5 Base, 200M parameters), inference only - no
  training or fine-tuning.
- Chronos is univariate: each of the {N_ASSETS} M6 assets was forecast as its own
  independent series (batched only for speed, batch size {SERIES_BATCH_SIZE}).
- Round: M6 Round {ROUND_NUMBER}, forecast origin {ORIGIN_DATE}.
- Context: {CONTEXT_LENGTH} weekday daily log returns,
  {metadata['context_start_date']} to {metadata['context_end_date']}, taken
  unchanged from the Stage 3 file `round_{ROUND_NUMBER:02d}_context.csv`
  (SHA-256 `{CONTEXT_SHA256[:16]}...`). No preprocessing was repeated: no
  normalising, standardising, smoothing, clipping, averaging or NaN-filling.
  CARR's and OGN's genuine leading missing values were left as NaN and handled
  by Chronos's missing-value mask.
- Horizon: {PREDICTION_LENGTH} weekdays, {FORECAST_DATES[0].date()} to {FORECAST_DATES[-1].date()}.
- Trajectories: {NUM_SAMPLES} sampled paths per asset; random seed {RANDOM_SEED}.
- Runtime: Google Colab GPU ({torch.cuda.get_device_name(0)}), dtype {DTYPE},
  chronos-forecasting {metadata['chronos_forecasting_version']}, torch {torch.__version__}.

## Result

- Raw output shape: {tuple(forecast_samples.shape)} = (assets, sampled
  trajectories, forecast weekdays) - exactly as specified.
- Validation passed: shape correct, all values finite, asset order preserved,
  saved file reloaded successfully with identical values and ordering.
- Raw forecasts saved to: `{SAMPLES_PATH}`
- Metadata saved to: `{METADATA_PATH}`

## Scope

The raw sampled trajectories were saved before any M6 post-processing. No
four-week return conversion, asset ranking, quintile assignment, quintile
probability construction, realised-outcome comparison or RPS evaluation was
performed in this notebook.
"""

REPORT_PATH.write_text(report, encoding="utf-8")
print(f"Report written to: {REPORT_PATH}")
print("Copy it into the repository as reports/chronos_t5_base_round01_inference_report.md")

Report written to: /content/drive/MyDrive/HonoursResearch/Round_1_Context/outputs/chronos_t5_base/chronos_t5_base_round01_inference_report.md
Copy it into the repository as reports/chronos_t5_base_round01_inference_report.md


## 10. Inference summary

A final one-screen confirmation of what this run produced.

In [15]:
print("Chronos-T5 Base - M6 Round 1 pilot")
print(f"  model            : {MODEL_ID}")
print(f"  context          : {metadata['context_start_date']} .. {metadata['context_end_date']} "
      f"({CONTEXT_LENGTH} weekdays, {N_ASSETS} assets)")
print(f"  forecast horizon : {FORECAST_DATES[0].date()} .. {FORECAST_DATES[-1].date()} "
      f"({PREDICTION_LENGTH} weekdays)")
print(f"  trajectories     : {NUM_SAMPLES} per asset")
print(f"  raw output shape : {forecast_samples.shape}")
print(f"  saved to         : {SAMPLES_PATH}")
print(f"  save verified    : {SAVE_VERIFIED}")
print("  post-processing  : none (no four-week returns, quintiles, or RPS)")
print("\nNext stage (separate notebook): convert these raw trajectories into M6 "
      "four-week returns, quintile probabilities and RPS.")

Chronos-T5 Base - M6 Round 1 pilot
  model            : amazon/chronos-t5-base
  context          : 2020-03-19 .. 2022-03-04 (512 weekdays, 100 assets)
  forecast horizon : 2022-03-07 .. 2022-04-01 (20 weekdays)
  trajectories     : 100 per asset
  raw output shape : (100, 100, 20)
  saved to         : /content/drive/MyDrive/HonoursResearch/Round_1_Context/outputs/chronos_t5_base/chronos_t5_base_round01_samples.npz
  save verified    : True
  post-processing  : none (no four-week returns, quintiles, or RPS)

Next stage (separate notebook): convert these raw trajectories into M6 four-week returns, quintile probabilities and RPS.
